In [1]:
%useLatestDescriptors
%use dataframe
%use lets-plot
%use lets-plot-gt

In [2]:
USE{
    repositories {
        maven {
            url = "https://repo.osgeo.org/repository/geotools-releases/"
        }
    }
    dependencies {
        implementation("org.geotools:gt-shapefile:32.1")
        implementation("org.geotools:gt-cql:32.1")
    }
}

In [ ]:
:classpath

In [4]:
data class SeawaterInformationByObservationPoint(
    val sta_cde: String,
    val sta_nam_kor: String,
    val obs_datetime: String,
    val obs_lay: String,
    val wtr_tmp: String,
    val dox: String?,
    val sal: String?,
    val gru_nam: String,
    val lon: Double,
    val lat: Double,
)

In [5]:
import java.io.File
import org.geotools.api.data.FileDataStore
import org.geotools.api.data.FileDataStoreFinder
import org.geotools.api.feature.simple.SimpleFeatureType
import org.geotools.feature.simple.SimpleFeatureTypeBuilder
import org.geotools.referencing.crs.DefaultGeographicCRS
import org.geotools.feature.simple.SimpleFeatureBuilder
import org.geotools.feature.simple.SimpleFeatureImpl
import org.geotools.data.collection.ListFeatureCollection
import org.geotools.geometry.jts.JTSFactoryFinder
import org.geotools.data.simple.SimpleFeatureCollection
import org.locationtech.jts.geom.Point

In [6]:
var dataStore: FileDataStore = FileDataStoreFinder.getDataStore(File("/Volumes/WorkSpace/Notebook/data/south_korea_Boundary.shp"))
val koreaFeatures:SimpleFeatureCollection = dataStore.featureSource.features
val korea = koreaFeatures.toSpatialDataset(10)

fun createFeatureType(): SimpleFeatureType {
    val builder = SimpleFeatureTypeBuilder()
    builder.setName( "Location" )

    builder.add("the_geom", org.locationtech.jts.geom.Point::class.java, DefaultGeographicCRS.WGS84) // 경도, 위도
    builder.add("wtr_tmp", String::class.java)
    builder.add("obs_datetime", String::class.java)
    builder.add("sta_nam_kor", String::class.java)
    builder.add("sta_cde", String::class.java)
    builder.add("tempBoundary",String::class.java)

    return builder.buildFeatureType()
}


fun createFeatureCollection(seaWaterInfo: List<SeawaterInformationByObservationPoint>): SimpleFeatureCollection {
    val featureType = createFeatureType()
    val featureBuilder = SimpleFeatureBuilder(featureType)
    val featureCollection = ListFeatureCollection(featureType)
    val geometryFactory = JTSFactoryFinder.getGeometryFactory()

    seaWaterInfo.forEach { info ->

        val point: Point = geometryFactory.createPoint(org.locationtech.jts.geom.Coordinate(info.lon, info.lat))
        featureBuilder.add(point)
        featureBuilder.add(info.wtr_tmp)
        featureBuilder.add(info.obs_datetime)
        featureBuilder.add(info.sta_nam_kor)
        featureBuilder.add(info.sta_cde)
        featureBuilder.add(
            when(info.wtr_tmp.toFloat()) {
                in 0.0..6.0 -> "Low"
                in 6.1..10.0 -> "Middle"
                in 10.1..16.0 -> "High"
                else -> "Unknown"
            })

        val feature = featureBuilder.buildFeature(null)
        featureCollection.add(feature)
    }
    return featureCollection
}

In [7]:
val df_current = DataFrame.readJson("http://127.0.0.1:7788/nifs/seawaterinfo/current")
df_current.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,median,max
sta_cde,String,67,42,0,bgj8a,3,null,null,bgj8a,fnm5b,wn087
sta_nam_kor,String,67,42,0,기장,3,null,null,강릉,완도 감목,해남 화산
obs_datetime,String,67,1,0,2025-04-02 09:00:00,67,null,null,2025-04-02 09:00:00,2025-04-02 09:00:00,2025-04-02 09:00:00
obs_lay,String,67,3,0,1,42,null,null,1,1,3
wtr_tmp,String,67,32,0,10,7,null,null,10,11.6,9.9
dox,String?,67,14,52,9.2,3,null,null,10.2,13,9.5
sal,String?,67,3,65,34.9,1,null,null,32.4,32.4,34.9
gru_nam,String,67,3,0,남해,36,null,null,남해,남해,서해
lon,Double,67,42,0,129.227000,3,127.578440,1.191157,124.729500,127.065400,129.549700
lat,Double,67,42,0,35.187000,3,35.434866,1.304027,33.310400,34.803800,38.368100


In [8]:
val dfCurrentList = df_current.filter {
    obs_lay.equals("1")
}.toListOf<SeawaterInformationByObservationPoint>()

In [9]:
fun createFeatureType(): SimpleFeatureType {
    val builder = SimpleFeatureTypeBuilder()
    builder.setName( "Location" )

    builder.add("the_geom", org.locationtech.jts.geom.Point::class.java, DefaultGeographicCRS.WGS84) // 경도, 위도
    builder.add("wtr_tmp", String::class.java)
    builder.add("obs_datetime", String::class.java)
    builder.add("sta_nam_kor", String::class.java)
    builder.add("sta_cde", String::class.java)
    builder.add("tempBoundary",String::class.java)

    return builder.buildFeatureType()
}


fun createFeatureCollection(seaWaterInfo: List<SeawaterInformationByObservationPoint>): SimpleFeatureCollection {
    val featureType = createFeatureType()
    val featureBuilder = SimpleFeatureBuilder(featureType)
    val featureCollection = ListFeatureCollection(featureType)
    val geometryFactory = JTSFactoryFinder.getGeometryFactory()

    seaWaterInfo.forEach { info ->

        val point: Point = geometryFactory.createPoint(org.locationtech.jts.geom.Coordinate(info.lon, info.lat))
        featureBuilder.add(point)
        featureBuilder.add(info.wtr_tmp)
        featureBuilder.add(info.obs_datetime)
        featureBuilder.add(info.sta_nam_kor)
        featureBuilder.add(info.sta_cde)
        featureBuilder.add(
            when(info.wtr_tmp.toFloat()) {
                in 0.0..6.0 -> "Low"
                in 6.1..10.0 -> "Middle"
                in 10.1..16.0 -> "High"
                else -> "Unknown"
            })

        val feature = featureBuilder.buildFeature(null)
        featureCollection.add(feature)
    }
    return featureCollection
}

In [10]:

val pointData = createFeatureCollection(dfCurrentList).toSpatialDataset(10)

In [11]:
val southKoreaBounds = koreaFeatures.bounds
southKoreaBounds.expandBy(0.5)
val southKoreaLimits = coordMap(
    xlim = southKoreaBounds.minX to southKoreaBounds.maxX,
    ylim = southKoreaBounds.minY to southKoreaBounds.maxY
)

In [12]:
val voidTheme = theme( panelBackground ="blank")

letsPlot() +
        geomMap(map = korea, fill = "#e5f5e0") +
        geomPoint(
            map=pointData ,
            size = 3,
            shape = 1,
            tooltips = layerTooltips()
                .anchor("top_right")
                .line("수집시간|@obs_datetime")
                .line("관측지점|@sta_nam_kor/@sta_cde")
                .line("온도|@wtr_tmp °C" )
        ){ color="tempBoundary" } +
        labs( title="Korea EastSea 수온 정보", color="수온범위", caption="Nifs") +
        southKoreaLimits + voidTheme +
        ggsize(800, 800)

<path d="M234.94951012350975 599.5085405476284 L234.94951012350975 599.5085405476284 L235.97690856377085 601.0543037432767 L237.2540614223708 602.0532438864589 L241.43832432897216 603.7164868013424 L242.7084210695666 605.0070868271941 L243.56220821791067 608.1206910310175 L244.40188328631484 609.3387648693879 L245.58730686231502 607.9768797265533 L246.24352344356157 609.3260774399105 L244.40188328631484 610.7511860302116 L244.99459507431675 612.8778576513464 L243.80917148964363 614.716622666378 L242.24491645640592 616.526774873083 L239.29896447756983 618.8817981503871 L237.08151793182515 623.1355809128399 L236.44261069591994 625.4036320039945 L233.47905173857907 626.0875082386506 L231.20164324617508 625.8867989768069 L227.8766314977929 628.5155025339427 L219.11339635997138 630.1273246002206 L216.18237100503575 632.6214721819151 L212.59352105031212 633.4199509862874 L208.7508654343328 633.0353739574311 L204.35953058068299 633.8546789055135 L200.70885340622772 632.756359190631 L192.4692117040413 632.8745647609453 L189.89952054467904 634.0780002513152 L188.36945103127255 636.7906158929168 L186.97234463482528 637.0815369286947 L185.57523831641265 636.4111396003859 L183.60128222766252 633.2110598121085 L180.97180254084742 632.3165214006563 L178.4203603257938 629.6411441242867 L177.53129256142347 626.3829943618471 L179.07657690704036 622.1608480850978 L180.53718809673046 619.5421267201755 L184.45331960301883 616.8339083984752 L187.02701505439836 612.3562966737459 L189.39307800917413 611.7493567621791 L190.48713193001458 609.383949431844 L203.38998343809726 605.4435346891464 L205.60569189871967 603.5324874471007 L214.0593969404781 602.6457181587912 L219.81348323130806 600.1342217415913 L221.1915667695739 600.5244181153607 L232.2611230255061 599.055179934343 L234.94951012350975 599.5085405476284 ZM275.900281180262 547.1199076402295 L275.900281180262 547.1199076402295 L275.9637860515413 549.6193988583086 L277.0645364716165 551.449925512221 L278.1935112336487 552.0756235752583 L278.31346484086316 552.9481221186502 L275.900281180262 552.0756235752583 L275.201727943022 550.1600820052563 L274.60901623305654 549.5981114057176 L274.5455113617754 547.5415065814232 L275.900281180262 547.1199076402295 ZM88.76564628786582 548.1802548371029 L88.76564628786582 548.1802548371029 L88.4975148008507 548.6486409487884 L88.11648574659012 548.5762555527926 L84.84245873293912 546.0423968254618 L84.63077592116497 545.2288534669865 L85.71741425263099 543.5120743335165 L86.677042850215 543.8358596571466 L87.43910080266687 544.7815856821403 L88.76564628786582 548.1802548371029 ZM212.79764402116234 535.8954467402255 L212.79764402116234 535.8954467402255 L215.4930714134589 535.8954467402255 L216.96779483915452 537.1191521492992 L216.7561121054132 537.5838601825135 L216.0857832794918 537.2044215596388 L215.93054913428386 537.6137028681769 L214.00423581244468 537.0722537008783 L213.24217785999645 538.091173912746 L210.10927266883846 540.3161901201524 L208.45109091204722 540.2437368358687 L207.52674277442202 539.0162007816389 L207.46323790314636 537.4303812904659 L208.22529593362924 535.874126383685 L209.76352415257134 534.7995110827419 L211.0830135197666 534.7995110827419 L212.79764402116234 535.8954467402255 ZM221.25084307698853 541.087569057358 L221.25084307698853 541.087569057358 L220.39705585060983 541.3219527525566 L218.93644466091973 540.1414486115109 L218.74593013379308 538.7348664041579 L219.48681987096097 534.9700936192007 L218.74593013379308 533.95507880274 L219.5079882423106 532.9058233922165 L221.71654520913944 533.6693183039256 L221.92117190290992 534.6928952714329 L221.09560908785352 535.4050642378093 L220.94743106065107 536.2024549416087 L222.47154711295116 538.2105376698605 L221.81533053170642 538.585670315978 L221.25084307698853 541.087569057358 ZM243.32230107284704 536.0702722075498 L243.32230107284704 536.0702722075498 L242.77192586281126 537.0466726909144 L241.75584852062275 537.5966498783469 L240.46458356474614 537.7074939658082 L238.72878469